In [28]:
import os
import re
from dotenv import load_dotenv
from operator import itemgetter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

load_dotenv()

True

# Connect to ChromaDB

In [2]:
api_key = os.getenv("API_KEY")
llm = model = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

# Initialize retriever

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

# Utility chains

Utility chains are pre-build for speceficic tasks. Examples:
- RetrievalQA
- AnalyzeDocumentChain

In [4]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [9]:
qa_chain.invoke("what is a Morpheme?")

{'query': 'what is a Morpheme?',
 'result': 'A morpheme is the smallest meaningful unit of language. It cannot be divided into smaller meaningful parts.\n\nFor example:\n*   The word "fox" consists of one morpheme: "fox".\n*   The word "cats" consists of two morphemes: "cat" (the root) and "-s" (an affix indicating plural).'}

In [14]:
qa_chain.invoke("what does BPE mean?")

{'query': 'what does BPE mean?',
 'result': 'BPE stands for Byte Pair Encoding.'}

# Foundational chains

Foundational chains can be:
- TransformChain: for transforming the data
- LLMChain: for using an llm with a prompt template -> Deprecated: Use RunnableSequence, e.g. ``prompt | llm`` instead

In [22]:
def limpiar_texto(texto: str) -> str:
    # Eliminamos los emojis utilizando un amplio rango unicode
    # Ten en cuenta que esto podría potencialmente eliminar algunos caracteres válidos que no son en inglés
    patron_emoji = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticonos
        "\U0001F300-\U0001F5FF"  # símbolos y pictogramas
        "\U0001F680-\U0001F6FF"  # símbolos de transporte y mapas
        "\U0001F1E0-\U0001F1FF"  # banderas (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE,
    )
    texto = patron_emoji.sub(r'', texto)

    # Removemos las URLs
    patron_url = re.compile(r'https?://\S+|www\.\S+')
    texto = patron_url.sub(r'', texto)

    return texto

In [23]:
cadena_que_limpia = RunnableLambda(limpiar_texto)

In [24]:
cadena_que_limpia.invoke('Check this page https://twitter.com/home 🙈')

'Check this page  '

In [7]:
plantilla = """Parafrasea este texto:

{texto_limpio}

En el estilo de una persona informal de {estilo}.

Parafraseado: """

prompt = PromptTemplate(
    input_variables=["texto_limpio", "estilo"],
    template=plantilla
)

In [8]:
cadena_que_cambia_estilo = prompt | llm

In [20]:
cadena_que_cambia_estilo.invoke(input={
    "texto_limpio": "Hola, ¿qué tal se encuentra usted?",
    "estilo": "Argentino",
})

AIMessage(content='¡Che, ¿cómo andás?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019efe87-24c6-7821-83ea-78321f279b99-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 8, 'total_tokens': 45, 'input_token_details': {'cache_read': 0}})

# Example of sequential chain

For executing a list of steps. The simple version has one input and one output in each step. The complex version can manage multiple inputs and outputs.

In [13]:
texto_entrada = """
¡Monterrey es una ciudad impresionante! 🏙️
Es conocida por su impresionante paisaje de montañas ⛰️ y su vibrante cultura norteña.
¡No olvides visitar el famoso Museo de Arte Contemporáneo (MARCO)!
🖼️ Si eres fanático del fútbol, no puedes perderte un partido de los Rayados o de los Tigres. ⚽
Aquí te dejo algunos enlaces para que puedas conocer más sobre esta maravillosa ciudad:
https://visitamonterrey.com, https://museomarco.org, https://rayados.com, https://www.tigres.com.mx.
¡Monterrey te espera con los brazos abiertos! 😃🇲🇽
"""


In [31]:
cadena_secuencial = (
    {
        # itemgetter extracts the string, then pipes (|) it to the cleaning chain
        "texto_limpio": itemgetter("texto") | cadena_que_limpia,
        "estilo": itemgetter("estilo")
    }
    | cadena_que_cambia_estilo
)

In [33]:
resultado = cadena_secuencial.invoke({
    "texto": texto_entrada,
    "estilo": "Argentino"
})

resultado

AIMessage(content='¡Che, loco! Monterrey es una locura, ¡una barbaridad!\n\nTiene unas montañas que te dejan con la boca abierta y una onda norteña que te vuela la peluca.\n\n¡Ojo! No te podés ir sin darte una vuelta por el MARCO, ¡un caño de museo!\n\nY si te va el fútbol, ¡agarrate! Ver un partido de los Rayados o los Tigres es un espectáculo aparte.\n\nAcá te dejo unas páginas para que te empapes de todo lo que tiene esta joyita:\n\n¡Monterrey te va a recibir con los brazos abiertos, papá!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f00f1-c634-75b3-a747-5bd4df1dff8b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 132, 'total_tokens': 259, 'input_token_details': {'cache_read': 0}})